# **Mixture of Experts (MoE): Practice**

### Practice for the lesson “Mixture of Experts (MoE)” of the module “Nonlinear interpretable models”

In the lesson “Mixture of Experts (MoE)” we said several things about MoE that are worth checking on data rather than taking on trust:

1. with the mixture likelihood (formula 1.3 of Jacobs, Jordan, Nowlan, and Hinton, 1991) the experts specialize on their regions of the data, and with the error of the overall prediction (formula 1.1) they cooperate;
2. the gate learns to predict the responsibility of an expert $h_k(x,y)$, but it sees only $x$, so its weight $g_k(x)$ is softer where regions meet;
3. routing tells who is responsible for an object, and why the prediction is what it is — that is told by the expert itself;
4. how often an expert is selected is not its importance, and importance is measured by removing the expert;
5. in sparse top-$k$ routing (Shazeer et al., 2017) a zero weight means “did not pass the selection”, and an even load of the experts is set by the load-balancing loss, not by the data.

We check all of it on synthetic data: two features, three regions with different laws, linear experts. The data is created in the notebook itself, the seeds are fixed, and the whole notebook runs on a CPU in a couple of minutes.

### Which libraries we use and why not others

| Library | What it has | What we take |
|---|---|---|
| `torch` | `nn.Linear` layers, `softmax`, `topk`, automatic differentiation | the whole MoE: experts, gate, both losses, and noisy top-$k$ — from the formulas of the lesson “Mixture of Experts (MoE)”, a few lines for each part |
| `matplotlib` | plots | the routing map and the “frequency versus importance” plot |
| `scikit-learn` | `GaussianMixture` — a mixture of distributions $p(x)$, with no experts and no gate | not used: MoE models $p(y \mid x)$, and scikit-learn has no ready MoE |
| `transformers` | MoE layers inside large language models: Mixtral, Switch Transformers | not used: routing there is visible only on a whole language model, and we need data where the regions and their laws are known in advance |

`torch` and `matplotlib` are already installed in Colab, nothing needs to be installed.

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

T0 = time.time()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SIGMA = 0.5            # σ in the expert likelihood N(y; f_k(x), σ²), formula (1.3)
STEPS, LR = 3000, 0.05 # full batch, Adam
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd', '#8c564b', '#17becf', '#bcbd22', '#7f7f7f']
print('device:', DEVICE, '| torch', torch.__version__)

## 1. Data: three regions with different laws

The features $x = (x_1, x_2)$ are uniform on the square $[-1, 1]^2$. The region is set by $x_1$, and each region has its own law:

- region 0, $x_1 < 0.4$ (about 70% of objects): $y = 3(x_1 + 0.3)^2 + x_2$ — a parabola in $x_1$ with its vertex at $x_1 = -0.3$;
- region 1, $0.4 \le x_1 < 0.8$ (about 20%): $y = -2 + 2x_2$;
- region 2, $x_1 \ge 0.8$ (about 10%): $y = 4 - 2x_2$.

Noise $\mathcal N(0, 0.1^2)$ is added to the answer. The training set has 3000 objects, the test set 2000, each with its own seed.

A linear expert describes the laws of regions 1 and 2 exactly, but a single line cannot describe the parabola of region 0. So it is not known in advance how many experts region 0 will get: this is what we look at.

In [ ]:
B1, B2 = 0.4, 0.8      # region boundaries along x1

def make_data(n, seed):
    """x ~ U[-1,1]^2; three regions along x1, each with its own law y(x), noise N(0, 0.1^2)."""
    g = torch.Generator().manual_seed(seed)
    x = torch.rand(n, 2, generator=g) * 2 - 1
    region = (x[:, 0] >= B1).long() + (x[:, 0] >= B2).long()
    laws = torch.stack([3 * (x[:, 0] + 0.3) ** 2 + x[:, 1],   # region 0: a parabola in x1
                        -2 + 2 * x[:, 1],                      # region 1
                        4 - 2 * x[:, 1]], 1)                   # region 2
    y = laws.gather(1, region[:, None])[:, 0] + 0.1 * torch.randn(n, generator=g)
    return x.to(DEVICE), y.to(DEVICE), region.to(DEVICE)

x_tr, y_tr, r_tr = make_data(3000, seed=1)
x_te, y_te, r_te = make_data(2000, seed=2)
print('shares of regions 0, 1, 2 in the training set:', [round(v, 3) for v in (torch.bincount(r_tr) / len(r_tr)).tolist()])

fig, ax = plt.subplots(figsize=(5.5, 4.4))
sc = ax.scatter(x_tr[:, 0].cpu(), x_tr[:, 1].cpu(), c=y_tr.cpu(), s=6, cmap='coolwarm')
for b in (B1, B2):
    ax.axvline(b, color='k', ls='--', lw=1)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$'); ax.set_title('Training set: color is y')
fig.colorbar(sc, label='$y$')
plt.show()

## 2. Classic MoE: two losses

The model is as in the lesson setup: $K = 4$ linear experts $f_k(x) = w_k^\top x + b_k$ and a linear gate $g(x) = \mathrm{softmax}(W_g x + c)$, the mixture prediction is $\hat y(x) = \sum_k g_k(x) f_k(x)$. At the start the weights of the gate are zero, and $g_k(x) = 1/4$ for every expert.

We compare two error functions from section 1 of Jacobs, Jordan, Nowlan, and Hinton (1991):

$$E_{1.1} = \Big(y - \sum_k g_k(x)\, f_k(x)\Big)^2, \qquad E_{1.3} = -\log \sum_k g_k(x)\, \exp\Big(-\frac{(y - f_k(x))^2}{2\sigma^2}\Big).$$

The paper uses $\sigma = 1$; we use $\sigma = 0.5$. It is a hyperparameter, and task 1 changes it.

Besides the mixture MSE we compute two more quantities on the test set. **MSE of a single expert** is the error if for every object we take the prediction of the single expert with the largest weight $g_k(x)$: for specialized experts it is not larger than the error of the mixture. **Expected error (1.2)** is the mean of $\sum_k g_k(x)\,(y - f_k(x))^2$: the error if the expert for an object is chosen at random with probabilities $g_k(x)$.

The table under each model gives the expert weights ($b$, $w_1$, $w_2$), the frequency (share of test objects where the expert has the largest weight $g_k(x)$) and the shares of objects of each region that the gate gives to this expert.

In [ ]:
class MoE(nn.Module):
    """K linear experts f_k(x) = w_k x + b_k and a linear gate g(x) = softmax(W_g x + c)."""
    def __init__(self, K):
        super().__init__()
        self.experts = nn.Linear(2, K)       # row k is expert k
        self.gate = nn.Linear(2, K)          # logits of the gate
        nn.init.zeros_(self.gate.weight)     # at the start g_k(x) = 1/K: all experts are equal
        nn.init.zeros_(self.gate.bias)

    def forward(self, x):
        return self.experts(x), F.log_softmax(self.gate(x), -1)   # predictions f_k(x) and log g_k(x)


def loss_mixture(f, log_g, y, sigma):
    """(1.3): -log sum_k g_k(x) N(y; f_k(x), sigma^2), without the constant term."""
    return -torch.logsumexp(log_g - (y[:, None] - f) ** 2 / (2 * sigma ** 2), -1).mean()


def loss_blend(f, log_g, y, sigma):
    """(1.1): squared error of the overall prediction sum_k g_k(x) f_k(x)."""
    return ((y - (log_g.exp() * f).sum(-1)) ** 2).mean()


def train(loss, K=4, seed=0, sigma=SIGMA, steps=STEPS, lr=LR):
    torch.manual_seed(seed)
    model = MoE(K).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        f, log_g = model(x_tr)
        opt.zero_grad()
        loss(f, log_g, y_tr, sigma).backward()
        opt.step()
    return model


@torch.no_grad()
def scores(model):
    """Test MSE: of the mixture, of the single expert with the largest g_k(x), and the expected error of one expert (1.2)."""
    f, log_g = model(x_te)
    g = log_g.exp()
    top = g.argmax(-1)
    return dict(mix=((g * f).sum(-1) - y_te).pow(2).mean().item(),
                top=(f.gather(1, top[:, None])[:, 0] - y_te).pow(2).mean().item(),
                e12=(g * (y_te[:, None] - f) ** 2).sum(-1).mean().item())


@torch.no_grad()
def describe(model, name):
    """Quality, expert weights and the share of objects of each region that the gate gives to the expert."""
    f, log_g = model(x_te)
    top = log_g.argmax(-1)
    K = f.shape[1]
    s = scores(model)
    print(f"{name}: mixture MSE {s['mix']:.3f}, MSE of the single expert with the largest weight {s['top']:.3f}, expected error (1.2) {s['e12']:.3f}")
    print('  expert    b      w1     w2   frequency  shares of regions 0, 1, 2')
    for k in range(K):
        b, w = model.experts.bias[k].item(), model.experts.weight[k].tolist()
        shares = [((top == k) & (r_te == r)).sum().item() / (r_te == r).sum().item() for r in range(3)]
        print(f'  {k}   {b:6.2f} {w[0]:6.2f} {w[1]:6.2f}   {(top == k).float().mean().item():.3f}   '
              + '  '.join(f'{v:.2f}' for v in shares))


mix = train(loss_mixture)
blend = train(loss_blend)
describe(mix, 'loss (1.3)')
print()
describe(blend, 'loss (1.1)')

If an expert has learned the law of its region, its row repeats that law: for region 1 it is $b = -2$, $w_1 = 0$, $w_2 = 2$, for region 2 — $b = 4$, $w_1 = 0$, $w_2 = -2$. Then the explanation has two levels, as in the lesson “Mixture of Experts (MoE)”: the gate says who is responsible for the object, and the expert weights say why the prediction is what it is. Find such rows for both models and look at how the parabola of region 0 is split.

One seed is one run. Let us repeat both losses on five seeds: the seed changes the initial weights of the experts, the data stays the same.

In [ ]:
print('seed  (1.3): mixture single (1.2)   (1.1): mixture single (1.2)')
for seed in range(5):
    a, b = scores(train(loss_mixture, seed=seed)), scores(train(loss_blend, seed=seed))
    print(f"  {seed}    {a['mix']:.3f}  {a['top']:.3f}  {a['e12']:.3f}     {b['mix']:.3f}  {b['top']:.3f}  {b['e12']:.3f}")

With which loss is the prediction of a single expert no worse than the mixture prediction on all five seeds? With which loss is the mixture more accurate while the individual experts err more? These are the specialization and the cooperation from section 1 of the paper.

## 3. Responsibility $h_k(x, y)$ versus gating weights $g_k(x)$

$$h_k(x, y) = \frac{g_k(x)\, \mathcal N\big(y;\, f_k(x),\, \sigma^2\big)}{\sum_j g_j(x)\, \mathcal N\big(y;\, f_j(x),\, \sigma^2\big)}$$

is the posterior probability of expert $k$ once the answer $y$ is known. The gate sees only $x$. We compare both quantities on the test set for the model with loss (1.3): on average over all objects and near the boundary between regions 1 and 2, where $|x_1 - 0.8| < 0.05$.

In [ ]:
@torch.no_grad()
def responsibility(model, x, y, sigma=SIGMA):
    """Responsibility h_k(x, y): posterior probability of expert k once the answer y is known."""
    f, log_g = model(x)
    return F.softmax(log_g - (y[:, None] - f) ** 2 / (2 * sigma ** 2), -1)


with torch.no_grad():
    f, log_g = mix(x_te)
g, h = log_g.exp(), responsibility(mix, x_te, y_te)
near = (x_te[:, 0] - B2).abs() < 0.05
print(f'mean of max_k g_k / max_k h_k over the test set: {g.max(-1).values.mean().item():.3f} / {h.max(-1).values.mean().item():.3f}')
print(f'the same near the boundary x1 = 0.8 (objects: {near.sum().item()}): {g[near].max(-1).values.mean().item():.3f} / {h[near].max(-1).values.mean().item():.3f}')
print(f'share of objects where argmax g and argmax h differ: {(g.argmax(-1) != h.argmax(-1)).float().mean().item():.4f}')
by_h = h.argmax(-1)
print(f'MSE of the single expert with the largest responsibility h_k: {(f.gather(1, by_h[:, None])[:, 0] - y_te).pow(2).mean().item():.3f}')

i = int(near.nonzero()[0])
print(f'\nOne object near the boundary: x = ({x_te[i, 0].item():.3f}, {x_te[i, 1].item():.3f}), y = {y_te[i].item():.2f}')
print('  g(x)   =', [round(v, 3) for v in g[i].tolist()])
print('  h(x,y) =', [round(v, 3) for v in h[i].tolist()])
print('  f(x)   =', [round(v, 2) for v in f[i].tolist()])

In the lesson, in the collapsible section “Rigorously: the gradients of the MoE loss”, we derived $\partial \ell / \partial s_k = g_k - h_k$, where $s_k$ are the logits of the gate. Let us check it with automatic differentiation on five objects: the difference must be at the level of float32 rounding error.

In [ ]:
x5, y5 = x_tr[:5], y_tr[:5]
s = mix.gate(x5).detach().requires_grad_(True)       # logits s_k(x) of the gate
f5 = mix.experts(x5).detach()
loss_mixture(f5, F.log_softmax(s, -1), y5, SIGMA).mul(len(x5)).backward()   # sum of the losses of five objects
gap = (s.grad - (s.softmax(-1) - responsibility(mix, x5, y5))).abs().max().item()
print(f'max |dl/ds_k - (g_k - h_k)| = {gap:.1e}')

## 4. Routing map

We color the square $[-1, 1]^2$ by the expert with the largest weight $g_k(x)$; the paler the color, the smaller that weight. The dashed lines are the true region boundaries $x_1 = 0.4$ and $x_1 = 0.8$. Under the map are the points on the line $x_2 = 0$ where the gate with loss (1.3) switches experts.

In [ ]:
xs = torch.linspace(-1, 1, 401, device=DEVICE)
X1, X2 = torch.meshgrid(xs, xs, indexing='xy')
grid = torch.stack([X1.flatten(), X2.flatten()], 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, (name, model) in zip(axes, [('Routing, loss (1.3)', mix), ('Routing, loss (1.1)', blend)]):
    with torch.no_grad():
        g = model.gate(grid).softmax(-1)
    K = g.shape[1]
    rgb = torch.tensor([[int(c[j:j + 2], 16) / 255 for j in (1, 3, 5)] for c in COLORS[:K]])
    img = rgb[g.argmax(-1).cpu()].reshape(401, 401, 3)
    alpha = g.max(-1).values.cpu().reshape(401, 401, 1)
    ax.imshow(torch.cat([img, alpha], -1), origin='lower', extent=(-1, 1, -1, 1))
    for b in (B1, B2):
        ax.axvline(b, color='k', ls='--', lw=1)
    ax.set_title(name); ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.legend(handles=[Patch(color=COLORS[k], label=f'expert {k}') for k in range(K)], loc='lower left', fontsize=8)
plt.tight_layout(); plt.show()

line = torch.stack([torch.linspace(-1, 1, 2001, device=DEVICE), torch.zeros(2001, device=DEVICE)], 1)
with torch.no_grad():
    a = mix.gate(line).argmax(-1)
print('Where the gate switches experts on the line x2 = 0:')
for j in (a[1:] != a[:-1]).nonzero()[:, 0].tolist():
    print(f'  x1 = {(line[j, 0] + line[j + 1, 0]).item() / 2:.3f}: expert {a[j].item()} -> expert {a[j + 1].item()}')

Compare the switching points with the region boundaries. Inside region 0 there is no true boundary, yet there is a switch: where is it relative to the vertex of the parabola $x_1 = -0.3$? The specialization of the experts does not have to coincide with the regions a person has set.

## 5. Removing an expert: selection frequency versus importance

The selection frequency of an expert is the share of test objects where it has the largest weight $g_k(x)$. We measure importance by intervention, as in the lesson “Mixture of Experts (MoE)”: we remove the expert (its logit becomes $-\infty$, softmax is taken over the rest) and see how much the mixture MSE on the test set grows.

In [ ]:
@torch.no_grad()
def mse_without(model, k, x, y):
    """Mixture MSE on (x, y) with expert k removed: its logit is -inf, softmax over the rest."""
    f, _ = model(x)
    logits = model.gate(x)
    logits[:, k] = float('-inf')
    return ((logits.softmax(-1) * f).sum(-1) - y).pow(2).mean().item()


@torch.no_grad()
def ablation_table(model, name):
    f, log_g = model(x_te)
    top = log_g.argmax(-1)
    base = scores(model)['mix']
    rows = []
    for k in range(f.shape[1]):
        freq = (top == k).float().mean().item()
        rows.append((freq, k, mse_without(model, k, x_te, y_te) - base))
    print(f'{name}: mixture MSE with all experts {base:.3f}')
    print('  expert  frequency  MSE increase without it')
    for freq, k, d in sorted(rows, reverse=True):
        print(f'  {k}       {freq:.3f}      {d:9.2f}')
    return rows


rows13 = ablation_table(mix, 'loss (1.3)')
print()
rows11 = ablation_table(blend, 'loss (1.1)')

fig, ax = plt.subplots(figsize=(5.5, 4))
for freq, k, d in rows13:
    ax.scatter(freq, d, s=80, color=COLORS[k])
    ax.annotate(f'expert {k}', (freq, d), textcoords='offset points', xytext=(6, 4))
ax.margins(0.2)
ax.set_xlabel('selection frequency (share of test objects)'); ax.set_ylabel('MSE increase without the expert'); ax.set_title('Loss (1.3): frequency versus importance')
plt.show()

Are the experts ordered by importance the same way as by frequency? And what do the huge numbers of the model with loss (1.1) mean?

## 6. Sparse top-$k$ gating (Shazeer et al., 2017)

In a transformer MoE the gate (router) keeps only $k$ experts for an object:

$$G(x) = \mathrm{softmax}\big(\mathrm{KeepTopK}(H(x), k)\big), \qquad H(x)_i = (x W_g)_i + \varepsilon_i \cdot \mathrm{softplus}\big((x W_{noise})_i\big), \qquad \varepsilon_i \sim \mathcal N(0, 1).$$

Noise is added only during training. The loss is the squared error of the overall prediction $\sum_i G_i(x) f_i(x)$, as in formula (1.1), plus the load-balancing loss $w_{importance} \cdot \mathrm{CV}(\mathrm{Importance})^2$, where $\mathrm{Importance}_i = \sum_x G_i(x)$ is the total weight of expert $i$ over the training set. We train such an MoE on the same synthetic data with $K = 4$ and $k = 2$ — without the load-balancing loss and with $w_{importance} = 1$. In the table, “in top-$k$” is the share of test objects where the expert passed the selection, Importance is its mean weight.

In [ ]:
class NoisyTopKMoE(nn.Module):
    """Noisy top-k gating (Shazeer et al., 2017) with linear experts."""
    def __init__(self, K, k):
        super().__init__()
        self.k = k
        self.experts = nn.Linear(2, K)
        self.gate = nn.Linear(2, K)          # x W_g
        self.noise = nn.Linear(2, K)         # x W_noise
        for layer in (self.gate, self.noise):
            nn.init.zeros_(layer.weight); nn.init.zeros_(layer.bias)

    def forward(self, x, noisy):
        H = self.gate(x)
        if noisy:                            # noise during training only
            H = H + torch.randn_like(H) * F.softplus(self.noise(x))
        top, idx = H.topk(self.k, -1)
        G = torch.full_like(H, float('-inf')).scatter(-1, idx, top).softmax(-1)   # KeepTopK, softmax
        return self.experts(x), G


def cv2(v):
    return v.var() / v.mean() ** 2           # squared coefficient of variation


def train_sparse(k, w_importance, K=4, seed=0, steps=STEPS, lr=LR):
    torch.manual_seed(seed)
    model = NoisyTopKMoE(K, k).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        f, G = model(x_tr, noisy=True)
        loss = ((y_tr - (G * f).sum(-1)) ** 2).mean() + w_importance * cv2(G.sum(0))
        opt.zero_grad(); loss.backward(); opt.step()
    return model


@torch.no_grad()
def describe_sparse(model, name):
    f, G = model(x_te, noisy=False)
    top = G.argmax(-1)
    print(f'{name}: mixture MSE {((G * f).sum(-1) - y_te).pow(2).mean().item():.3f}, '
          f'CV^2(Importance) = {cv2(G.sum(0)).item():.3f}, share of zero weights G {(G == 0).float().mean().item():.2f}')
    print('  expert  in top-k  Importance  shares of regions 0, 1, 2 (expert with the largest weight)')
    for e in range(G.shape[1]):
        shares = [((top == e) & (r_te == r)).sum().item() / (r_te == r).sum().item() for r in range(3)]
        print(f'  {e}     {(G[:, e] > 0).float().mean().item():.3f}    {G[:, e].mean().item():.3f}     '
              + '  '.join(f'{v:.2f}' for v in shares))


sparse0 = train_sparse(k=2, w_importance=0.0)
sparse1 = train_sparse(k=2, w_importance=1.0)
describe_sparse(sparse0, 'k = 2, no load-balancing loss (w_importance = 0)')
print()
describe_sparse(sparse1, 'k = 2, w_importance = 1')
print(f'\nnotebook run time up to this cell: {time.time() - T0:.0f} s')

Half of the weights $G$ are zero by construction: two of the four experts pass the selection, and a zero weight means “did not pass the selection”, not “useless for the object”. Compare the load of the experts without the load-balancing loss and with it: the data and the model are the same.

## Tasks

1. **σ in the likelihood.** Train an MoE with loss (1.3) at $\sigma = 1$ and $\sigma = 2$ instead of $0.5$. How many experts get at least 1% of the test objects, and how do the mixture MSE and the single-expert MSE change? Why does region 0 get fewer experts as $\sigma$ grows?
2. **More experts.** Train an MoE with loss (1.3) and $K = 8$. How many of the eight experts get at least 1% of the test objects, and what did they learn? Compare their weights with the experts of the model with $K = 4$.
3. **Removal per region.** For the model with loss (1.3) from section 2, compute the MSE increase without each expert separately on each of the three test regions. On which region and by how much does the quality drop without the rarest expert?
4. **$k = 1$ in noisy top-$k$.** Train `train_sparse` with $k = 1$. What did the gate learn (look at `model.gate.weight`), and where does it send the test objects? Explain it with the formula for $G(x)$. Then build the Switch Transformer routing (Fedus, Zoph, and Shazeer, 2022): $k = 1$, and the expert output is multiplied by the router probability $p_i(x) = \mathrm{softmax}(W_g x)_i$. Does the gate learn now?
5. **Strength of balancing.** Repeat `train_sparse` with $k = 2$ and $w_{importance} = 0,\ 0.1,\ 1,\ 10$. How do $\mathrm{CV}^2$, the load of the experts, and the mixture MSE change?

## Conclusions

- With the mixture likelihood (1.3) the experts specialize: on all five seeds the prediction of the single expert with the largest weight is more accurate than the mixture prediction, and the expert weights repeat the laws of their regions. With the error of the overall prediction (1.1) the mixture is more accurate, but the experts cooperate: individually their predictions are far from the answer, the weights resemble none of the laws, and such an expert cannot be read as “the law of its region”.
- Specialization does not have to coincide with the regions a person has set: two experts share the parabola of region 0, and the gate puts a boundary where there is no true boundary.
- The gate predicts responsibility from $x$ alone: far from the boundaries $g_k(x)$ almost coincides with $h_k(x, y)$, while near a boundary it stays soft, although $h_k(x, y)$ already knows the expert from the answer.
- Selection frequency is not importance: the most frequent expert is replaced by its neighbor on the parabola, while the rare one is irreplaceable on its region. With loss (1.1) the MSE grows several times without any expert, even without the one that has the largest weight on no object at all: the experts correct each other's errors.
- In top-$k$ routing a zero weight means “did not pass the selection”. The coefficient of the load-balancing loss changes the load of the experts on the same data, so an even load says nothing about the data. With $k = 1$ in the formula of Shazeer et al. the gate gets no gradient, and the Switch Transformer gets around it with the factor $p_i(x)$.